# Сбор постов через VK API

Выгружаем стену сообщества методом `wall.get`. Нужен сервисный или пользовательский токен и id сообщества в `.env`. Готовая выгрузка уже лежит в `../data/vk_posts_raw.xlsx`, так что этот шаг можно пропустить

In [ ]:
# Импорт библиотек
import os
import re
import time
from datetime import datetime, timezone

import pandas as pd
import requests
from dotenv import load_dotenv

In [ ]:
# Параметры подключения к API
# Токен и id сообщества берём из .env (шаблон лежит в .env.example), в коде их нет
load_dotenv()
TOKEN = os.getenv("VK_API_TOKEN")
GROUP_ID = os.getenv("VK_GROUP_ID")  # для сообщества id со знаком минус, например -123456
if not TOKEN or not GROUP_ID:
    raise RuntimeError("Задайте VK_API_TOKEN и VK_GROUP_ID в файле .env (см. .env.example)")

COUNT = 100          # максимум постов за один запрос wall.get
API_VERSION = "5.131"
url_wall = "https://api.vk.com/method/wall.get

In [ ]:
# Сколько всего постов на стене
response = requests.get(url_wall, params={
    "access_token": TOKEN, "v": API_VERSION, "owner_id": GROUP_ID, "count": 1,
}).json()
total_posts = response["response"]["count"]
print(f"Всего постов: {total_posts}")

In [ ]:
# Получение всех постов с пагинацией
def get_all_posts(total):
    all_posts = []
    for offset in range(0, total, COUNT):
        params = {
            "access_token": TOKEN,
            "v": API_VERSION,
            "owner_id": GROUP_ID,
            "count": COUNT,
            "offset": offset,
        }
        response = requests.get(url_wall, params=params).json()

        if "response" not in response:
            print("Ошибка:", response)
            break
        all_posts.extend(response["response"]["items"])
        time.sleep(0.35)  # у VK API лимит 3 запроса в секунду
    return all_posts

In [ ]:
# Из ответа API оставляем только нужные поля
# Время сохраняем в UTC, в московское его переводит 02_prepare_data.ipynb
def process_posts(posts):
    data = []
    for post in posts:
        data.append({
            "Post ID": post["id"],
            "Дата публикации": datetime.fromtimestamp(post["date"], tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
            "Просмотры": post.get("views", {}).get("count", 0),
            "Лайки": post["likes"]["count"],
            "Комментарии": post["comments"]["count"],
            "Текст поста": post["text"],
        })
    return data

In [ ]:
posts = get_all_posts(total_posts)
df = pd.DataFrame(process_posts(posts))

os.makedirs("../data", exist_ok=True)
df.to_excel("../data/vk_posts_raw.xlsx", index=False)
print(f"Сохранено {len(df)} постов в ../data/vk_posts_raw.xlsx")

In [ ]:
df.head()

In [ ]:
# Топ хештегов
hashtags = {}
for post in posts:
    for tag in re.findall(r"#(\w+)", post["text"]):
        hashtags[tag] = hashtags.get(tag, 0) + 1

print("ТОП-10 хештегов:")
for tag, count in sorted(hashtags.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"#{tag}: {count}")